In [1]:
import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [2]:
DATA_PATH = "PJME_hourly.csv"

df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])

df = df.sort_values("Datetime").drop_duplicates("Datetime")
df = df.dropna(subset=["PJME_MW"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["Datetime"].min(), "to", df["Datetime"].max())
print(df.head())

Dataset shape: (145362, 2)
Date range: 2002-01-01 01:00:00 to 2018-08-03 00:00:00
             Datetime  PJME_MW
0 2002-01-01 01:00:00  30393.0
1 2002-01-01 02:00:00  29265.0
2 2002-01-01 03:00:00  28357.0
3 2002-01-01 04:00:00  27899.0
4 2002-01-01 05:00:00  28057.0


In [3]:
LOOKBACK = 168
HORIZON = 24

values = df["PJME_MW"].to_numpy(dtype=np.float32)
timestamps = df["Datetime"].to_numpy()

N = len(values)
train_end = int(N * 0.70)
val_end = int(N * 0.85)

scaler = StandardScaler()
scaler.fit(values[:train_end].reshape(-1, 1))

scaled_values = scaler.transform(
    values.reshape(-1, 1)
).flatten().astype(np.float32)

print("Training rows:", train_end)
print("Validation rows:", val_end - train_end)
print("Testing rows:", N - val_end)

Training rows: 101753
Validation rows: 21804
Testing rows: 21805


In [4]:
class EnergyDataset(Dataset):
    def __init__(self, values, start_indices, lookback, horizon):
        self.values = values
        self.starts = start_indices
        self.lookback = lookback
        self.horizon = horizon

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        start = self.starts[idx]
        split = start + self.lookback

        x = self.values[start:split]
        y = self.values[split:split + self.horizon]

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y


# Start index refers to the first input hour.
all_starts = np.arange(0, N - LOOKBACK - HORIZON + 1)

target_start = all_starts + LOOKBACK
target_end = target_start + HORIZON

train_starts = all_starts[target_end <= train_end]
val_starts = all_starts[
    (target_start >= train_end) & (target_end <= val_end)
]
test_starts = all_starts[target_start >= val_end]

train_dataset = EnergyDataset(
    scaled_values, train_starts, LOOKBACK, HORIZON
)
val_dataset = EnergyDataset(
    scaled_values, val_starts, LOOKBACK, HORIZON
)
test_dataset = EnergyDataset(
    scaled_values, test_starts, LOOKBACK, HORIZON
)

print("Train sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Test sequences:", len(test_dataset))

Train sequences: 101562
Validation sequences: 21781
Test sequences: 21782


In [5]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("DataLoaders created.")

DataLoaders created.


In [6]:
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.2):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        )
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = self.relu(out + residual)
        return out


class DeepCNN(nn.Module):
    def __init__(self, horizon=24):
        super().__init__()

        self.block1 = ResidualConvBlock(1, 64)
        self.block2 = ResidualConvBlock(64, 128)
        self.block3 = ResidualConvBlock(128, 128)
        self.block4 = ResidualConvBlock(128, 256)

        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, horizon)
        )

    def forward(self, x):
        # Input: (batch, 168, 1)
        x = x.transpose(1, 2)  # (batch, 1, 168)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        avg = self.avg_pool(x).squeeze(-1)
        maximum = self.max_pool(x).squeeze(-1)

        x = torch.cat([avg, maximum], dim=1)
        return self.fc(x)


model = DeepCNN(horizon=HORIZON).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", f"{total_params:,}")

DeepCNN(
  (block1): ResidualConvBlock(
    (conv1): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (dropout): Dropout(p=0.2, inplace=False)
    (shortcut): Conv1d(1, 64, kernel_size=(1,), stride=(1,))
  )
  (block2): ResidualConvBlock(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (dropout): Dropout(p=0.2, inplace=False)
    (shortcut): Conv1d(64, 128, kernel_size=(1,), stride=(1,))
  )
  (block3

In [7]:
CHECKPOINT_DIR = "cnn_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")
LAST_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
LOG_PATH = os.path.join(CHECKPOINT_DIR, "training_log.csv")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 40
PATIENCE = 8

best_val_loss = float("inf")
start_epoch = 0
epochs_without_improvement = 0
history = []

if os.path.exists(LAST_PATH):
    checkpoint = torch.load(
        LAST_PATH, map_location=device, weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    epochs_without_improvement = checkpoint["patience_counter"]

    if os.path.exists(LOG_PATH):
        history = pd.read_csv(LOG_PATH).to_dict("records")

    print("Resuming from epoch:", start_epoch + 1)
else:
    print("Starting training from scratch.")

Starting training from scratch.


In [ ]:
def run_epoch(loader, training):
    model.train() if training else model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            predictions = model(x)
            loss = criterion(predictions, y)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples


for epoch in range(start_epoch, EPOCHS):
    start_time = time.time()

    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(model.state_dict(), BEST_PATH)
    else:
        epochs_without_improvement += 1

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "patience_counter": epochs_without_improvement
    }, LAST_PATH)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    pd.DataFrame(history).to_csv(LOG_PATH, index=False)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train: {train_loss:.5f} | "
        f"Val: {val_loss:.5f} | "
        f"Time: {elapsed:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

print("Training completed.")
print("Best validation loss:", best_val_loss)

Epoch 1/40 | Train: 0.28304 | Val: 0.13103 | Time: 58.5s
Epoch 2/40 | Train: 0.17325 | Val: 0.12145 | Time: 56.2s
Epoch 3/40 | Train: 0.16030 | Val: 0.12979 | Time: 56.9s
Epoch 4/40 | Train: 0.15347 | Val: 0.12536 | Time: 55.9s
Epoch 5/40 | Train: 0.14670 | Val: 0.11659 | Time: 55.9s
Epoch 6/40 | Train: 0.14391 | Val: 0.12169 | Time: 56.0s
Epoch 7/40 | Train: 0.14066 | Val: 0.11477 | Time: 58.4s
Epoch 8/40 | Train: 0.13894 | Val: 0.11661 | Time: 59.0s
Epoch 9/40 | Train: 0.13757 | Val: 0.11732 | Time: 57.0s
Epoch 10/40 | Train: 0.13528 | Val: 0.12359 | Time: 59.1s
Epoch 11/40 | Train: 0.13228 | Val: 0.12245 | Time: 57.7s
Epoch 14/40 | Train: 0.12825 | Val: 0.12685 | Time: 57.0s
Epoch 15/40 | Train: 0.12687 | Val: 0.12590 | Time: 58.0s
Epoch 16/40 | Train: 0.12636 | Val: 0.11838 | Time: 60.3s
Epoch 17/40 | Train: 0.12535 | Val: 0.11860 | Time: 59.7s
Epoch 18/40 | Train: 0.12433 | Val: 0.12006 | Time: 57.9s


In [9]:
import torch

checkpoint = torch.load(
    "cnn_checkpoints/last_checkpoint.pth",
    map_location="cpu",
    weights_only=False
)

print("Last completed epoch:", checkpoint["epoch"] + 1)
print("Best validation loss:", checkpoint["best_val_loss"])

Last completed epoch: 20
Best validation loss: 0.11021740214582684


In [10]:
import os
import torch
import pandas as pd

LAST_PATH = "cnn_checkpoints/last_checkpoint.pth"
LOG_PATH = "cnn_checkpoints/training_log.csv"

# Check the saved checkpoint
if os.path.exists(LAST_PATH):
    checkpoint = torch.load(
        LAST_PATH,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint epoch:", checkpoint["epoch"] + 1)
    print("Best validation loss:", checkpoint["best_val_loss"])
    print("Patience counter:", checkpoint["patience_counter"])
    print("Checkpoint modified:", os.path.getmtime(LAST_PATH))

# Check the training log
if os.path.exists(LOG_PATH):
    log = pd.read_csv(LOG_PATH)

    print("\nLast logged epoch:", log["epoch"].max())
    print("\nLast 5 training records:")
    print(log.tail(5).to_string(index=False))

Checkpoint epoch: 20
Best validation loss: 0.11021740214582684
Patience counter: 8
Checkpoint modified: 1790524576.9821343

Last logged epoch: 20

Last 5 training records:
 epoch  train_loss  val_loss
    16    0.126355  0.118385
    17    0.125347  0.118595
    18    0.124327  0.120063
    19    0.122797  0.121886
    20    0.121842  0.119625


In [11]:
model.load_state_dict(
    torch.load(
        "cnn_checkpoints/best_model.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Best model loaded successfully.")

Best model loaded successfully.


In [12]:
all_predictions = []
all_actuals = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        predictions = model(x)

        all_predictions.append(predictions.cpu().numpy())
        all_actuals.append(y.numpy())

pred_scaled = np.concatenate(all_predictions, axis=0)
actual_scaled = np.concatenate(all_actuals, axis=0)

# Convert back to MW
predictions = scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(-1, HORIZON)

actuals = scaler.inverse_transform(
    actual_scaled.reshape(-1, 1)
).reshape(-1, HORIZON)

print("Predictions shape:", predictions.shape)
print("Actuals shape:", actuals.shape)

Predictions shape: (21782, 24)
Actuals shape: (21782, 24)


In [13]:
mae = mean_absolute_error(
    actuals.flatten(), predictions.flatten()
)

rmse = np.sqrt(mean_squared_error(
    actuals.flatten(), predictions.flatten()
))

wape = (
    np.sum(np.abs(actuals - predictions))
    / np.sum(np.abs(actuals))
) * 100

print("\nDeep 1D-CNN Test Results")
print("-" * 35)
print(f"MAE  : {mae:.2f} MW")
print(f"RMSE : {rmse:.2f} MW")
print(f"WAPE : {wape:.2f}%")


Deep 1D-CNN Test Results
-----------------------------------
MAE  : 1696.57 MW
RMSE : 2255.89 MW
WAPE : 5.46%


In [14]:
rows = []

for i, start in enumerate(test_starts):
    target_start = start + LOOKBACK

    for h in range(HORIZON):
        target_idx = target_start + h

        rows.append({
            "Forecast_Start": pd.Timestamp(timestamps[start + LOOKBACK - 1]),
            "Target_Datetime": pd.Timestamp(timestamps[target_idx]),
            "Horizon_Hour": h + 1,
            "Actual_MW": float(actuals[i, h]),
            "Predicted_MW": float(predictions[i, h])
        })

predictions_df = pd.DataFrame(rows)

predictions_df.to_csv(
    "cnn_test_predictions.csv", index=False
)

print("Saved: cnn_test_predictions.csv")
print(predictions_df.head())

Saved: cnn_test_predictions.csv
       Forecast_Start     Target_Datetime  Horizon_Hour  Actual_MW  \
0 2016-02-06 08:00:00 2016-02-06 09:00:00             1    33474.0   
1 2016-02-06 08:00:00 2016-02-06 10:00:00             2    32987.0   
2 2016-02-06 08:00:00 2016-02-06 11:00:00             3    32173.0   
3 2016-02-06 08:00:00 2016-02-06 12:00:00             4    31337.0   
4 2016-02-06 08:00:00 2016-02-06 13:00:00             5    30545.0   

   Predicted_MW  
0  31555.890625  
1  31686.226562  
2  31418.498047  
3  31079.816406  
4  30813.099609  
